In [1]:
# %pip install pygame

In [2]:
import os
import openai
from IPython.display import Audio,display

from openai import OpenAI
apiKey = os.getenv('OPENAI_API_KEY')
openai.api_key = apiKey 

import pygame



pygame 2.6.1 (SDL 2.28.4, Python 3.10.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
client = OpenAI()
conv_history=[{'role':'system','content':'You are a helpful teacher' }]


In [ ]:
def speak_text(text):
    response = client.audio.speech.create(
        model="tts-1",
        voice="nova",
        input=text
    )
    
    filename = 'out.mp3'
    with open('out.mp3','wb') as fp:
        fp.write( response.content)
    pygame.mixer.init()
    try:
        pygame.mixer.music.load(filename)
        pygame.mixer.music.play()
        
        while pygame.mixer.music.get_busy():
            pygame.time.Clock().tick(10)
            

        pygame.mixer.music.unload() # 현재 로드된 파일을 목록에서 제거
    finally:
        pygame.mixer.quit()

    
    if os.path.exists(filename):
        os.remove(filename)
    


In [5]:
def conv_with_gpt( user_input):
    conv_history.append( {'role':'user','content':user_input})    
    completion = client.chat.completions.create(model='gpt-3.5-turbo',
        messages=conv_history )
    
    chatReply = completion.choices[0].message.content
    conv_history.append( {'role':'assistant','content':chatReply} )
    return chatReply


In [6]:
while True :
    response=conv_with_gpt(input("질문"))
    speak_text(response)
    i=input('계속하시겠습니까(y/n)?')
    if i.upper() == 'N':
        break
    